# 🛡️ ChurnGuard — Model Training Notebook

This notebook covers the full pipeline:
1. Install dependencies
2. Load & explore the Telco Churn dataset
3. Preprocess data
4. Train XGBoost model
5. Evaluate model performance
6. Set up SHAP explainer
7. Export model artifacts

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn --quiet

## 📥 Step 2 — Load Dataset

We use the IBM Telco Customer Churn dataset.
You can download it from Kaggle or use the direct URL below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset directly from GitHub mirror
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

print('Shape:', df.shape)
df.head()

## 🔍 Step 3 — Exploratory Data Analysis

In [ ]:
# Basic info
df.info()

In [ ]:
# Check missing values
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Churn distribution
plt.figure(figsize=(5, 4))
df['Churn'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

churn_rate = df['Churn'].value_counts(normalize=True) * 100
print(f'Churn rate: {churn_rate["Yes"]:.1f}%')

In [ ]:
# Numeric feature distributions by churn
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    for churn_val, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
        subset = df[df['Churn'] == churn_val][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, label=churn_val, color=color)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Feature Distributions by Churn', y=1.02)
plt.tight_layout()
plt.show()

## ⚙️ Step 4 — Preprocessing

In [ ]:
# Drop customerID (not a feature)
df = df.drop(columns=['customerID'])

# Fix TotalCharges: it's stored as string, convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with 0 (new customers with 0 tenure)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Encode target variable
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Convert SeniorCitizen to string for consistency
df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)

print('Preprocessing done.')
print('Churn value counts:', df['Churn'].value_counts().to_dict())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Identify categorical and numeric columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols.remove('Churn')  # Remove target

print('Categorical columns:', cat_cols)
print('Numeric columns:', num_cols)

# Label encode all categorical columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('\nEncoding done. Final shape:', df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

# Save feature column order — CRITICAL for inference
feature_columns = X.columns.tolist()
print('Features:', feature_columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

## 🤖 Step 5 — Train XGBoost Model

In [ ]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

# Handle class imbalance via scale_pos_weight
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print('\nTraining complete!')

## 📊 Step 6 — Evaluate Model

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_pred_proba))
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {auc:.3f})', color='steelblue')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (XGBoost built-in)
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=feature_columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
feat_imp.head(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances (XGBoost)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 🔬 Step 7 — SHAP Explainability

In [ ]:
import shap

# Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(model)

# Compute SHAP values on test set (sample for speed)
X_sample = X_test.sample(200, random_state=42)
shap_values = explainer.shap_values(X_sample)

print('SHAP values shape:', shap_values.shape)

In [ ]:
# SHAP Summary Plot (Beeswarm)
shap.summary_plot(shap_values, X_sample, plot_type='dot', max_display=15)

In [ ]:
# SHAP for a single prediction (index 0)
single_input = X_test.iloc[[0]]
single_shap = explainer.shap_values(single_input)[0]

shap_dict = dict(zip(feature_columns, single_shap.tolist()))
print('SHAP values for first test sample:')
for k, v in sorted(shap_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:10]:
    print(f'  {k}: {v:.4f}')

## 💾 Step 8 — Export Model Artifacts

In [ ]:
import os
import json

# Create model directory
os.makedirs('model', exist_ok=True)

# Save XGBoost model in native JSON format
model.save_model('model/churn_model.json')
print('Model saved to model/churn_model.json')

# Save feature column order (critical for correct inference)
with open('model/features.json', 'w') as f:
    json.dump(feature_columns, f)
print('Features saved to model/features.json')

# Save label encoding map for categorical columns
# (Needed so the API can encode incoming string values correctly)
encoding_map = {}
for col in cat_cols:
    # Re-fit LabelEncoder per column on original df to capture mapping
    original_df = pd.read_csv(url)
    original_df['SeniorCitizen'] = original_df['SeniorCitizen'].astype(str)
    le_temp = LabelEncoder()
    le_temp.fit(original_df[col].astype(str))
    encoding_map[col] = {str(cls): int(i) for i, cls in enumerate(le_temp.classes_)}

with open('model/encoding_map.json', 'w') as f:
    json.dump(encoding_map, f, indent=2)
print('Encoding map saved to model/encoding_map.json')

In [ ]:
# Verify artifacts
print('Files in model/:')
for f in os.listdir('model'):
    size = os.path.getsize(f'model/{f}') / 1024
    print(f'  {f} ({size:.1f} KB)')

In [ ]:
# Quick reload test — verify the model reloads and predicts correctly
test_model = xgb.XGBClassifier()
test_model.load_model('model/churn_model.json')

with open('model/features.json') as f:
    loaded_features = json.load(f)

test_pred = test_model.predict_proba(X_test[loaded_features])[:, 1]
test_auc = roc_auc_score(y_test, test_pred)
print(f'Reloaded model ROC-AUC: {test_auc:.4f} ✅')

In [ ]:
# Download artifacts from Colab to your machine
from google.colab import files
import zipfile

with zipfile.ZipFile('churnguard_model_artifacts.zip', 'w') as zf:
    for fname in ['churn_model.json', 'features.json', 'encoding_map.json']:
        zf.write(f'model/{fname}', fname)

files.download('churnguard_model_artifacts.zip')
print('Download started!')

## ✅ Summary

| Artifact | Path | Purpose |
|---|---|---|
| `churn_model.json` | `model/` | XGBoost model weights |
| `features.json` | `model/` | Ordered feature list for correct inference |
| `encoding_map.json` | `model/` | Label encoding map for categorical inputs |

**Next step:** Place the `model/` folder in your ChurnGuard project root and build the FastAPI app!